# Dokumentasi singkat notebook — Laboratorium forensik LLM (TokoBaju.id)

Notebook ini merupakan **runtime demonstrasi** untuk kode di `source_code_forensik.py`. **Korpus 28 skenario** ada di **`dataset_eksperimen.py`** (impor `SERANGAN`). **Unggah kedua berkas di dua langkah terpisah**—satu sel untuk `source_code_forensik.py`, lalu sel berikutnya hanya untuk `dataset_eksperimen.py` (jangan memilih dua file sekaligus dalam satu dialog). Spesifikasi arsitektur, skema dataset, kontrak label forensik, dan daftar endpoint HTTP dirinci dalam **`DOKUMENTASI_PROYEK.md`**. **Metodologi forensik jaringan** (model bukti berlapis: log aplikasi + `request_id` + PCAP/HTTP) ada di **`METODOLOGI_FORENSIK_JARINGAN.md`** — unggah ke `/content` jika ingin rujukan teks di Colab.

## Ruang lingkup eksperimen

- **Target simulasi:** aplikasi *customer service* berbasis Flask yang memanggil model lokal melalui Ollama (default: `llama3`).
- **Korpus uji:** 28 skenario *prompt injection* dengan metadata `kategori` dan `kelas_ancaman` (stub taksonomi; pemetaan ke kerangka standar dilakukan secara manual di luar runtime).
- **Alur batch:** setiap sampel dikirim ke `/chat` → respons diklasifikasikan secara heuristik → agregat ditulis ke `laporan_forensik.json` → **ekspor dataset** ke `dataset/injection_runs.jsonl`, `dataset/injection_runs.csv`, dan `dataset/dataset_manifest.json`.

## Antarmuka publik lewat ngrok

Setelah tunnel aktif, UI tersedia pada path:

| Path | Fungsi |
|------|--------|
| `/` | Chat target dengan mitigasi tampilan teks (escape HTML di sisi klien). |
| `/dashboard` | Visualisasi agregat dan tabel per sampel. |
| `/dataset` | Indeks unduhan artefak dataset + pratinjau manifest. |
| `/exports/...` | Unduhan terkontrol (whitelist nama berkas). |

## Asumsi lingkungan Colab

Runtime Linux; proses `ollama serve` di belakang layar; koneksi keluar untuk unduhan model, paket Python, dan pembukaan sesi ngrok. **Kredensial ngrok tidak disimpan dalam repositori**; token dimasukkan melalui input interaktif di sel eksekusi (bukan tertulis di kode).

## Sel di bawah ini sebagai *infrastructure cells*

Kode pada sel-sel berikut persis mewakili perkakas build dan orkestrasi singkat; interpretasi hasil mengacu pada dokumentasi proyek dan berkas JSON/JSONL yang dihasilkan.

In [ ]:
# Instalasi lingkungan (Colab GPU opsional — Ollama pakai CPU juga boleh)
# tcpdump: opsi rekaman PCAP (bukti lalu lintas); jika memakan quota, bisa di-skip.
!apt-get update -qq && apt-get install -y -qq zstd curl tcpdump
!curl -fsSL https://ollama.com/install.sh | sh
%pip install -q ollama flask requests pyngrok

In [ ]:
# Jalankan daemon Ollama di background
import subprocess, time, os
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
print("Ollama serve dimulai.")

In [ ]:
# Unduh model (butuh beberapa menit)
!ollama pull llama3

In [ ]:
# Langkah 1 — unggah hanya source_code_forensik.py (satu berkas dalam dialog ini).
from google.colab import files
import os

up = files.upload()
path = "/content/source_code_forensik.py"
if not os.path.isfile(path):
    raise FileNotFoundError(
        f"Belum ada {path}. Pilih satu file: source_code_forensik.py lalu jalankan ulang sel ini."
    )
print("OK:", path, "| berkas di dialog:", list(up.keys()))

In [ ]:
# Langkah 2 — unggah hanya dataset_eksperimen.py (dialog terpisah dari langkah 1).
from google.colab import files
import os

up = files.upload()
path = "/content/dataset_eksperimen.py"
if not os.path.isfile(path):
    raise FileNotFoundError(
        f"Belum ada {path}. Pilih satu file: dataset_eksperimen.py lalu jalankan ulang sel ini."
    )
print("OK:", path, "| berkas di dialog:", list(up.keys()))

## Bukti lalu lintas jaringan (opsional, selaras judul penelitian)

Setiap respons `/chat` menyertakan **`request_id`** (UUID) yang sama di `forensic_log.txt`, `hasil_serangan.json`, dan `injection_runs.jsonl` — jodohkan dengan **frame HTTP** di berkas PCAP (waktu + POST `/chat`).

**Alur singkat:** (1) jalankan sel berikut untuk memulai `tcpdump` pada loopback port **5001**; (2) jalankan sel **orkestrasi ngrok + batch**; (3) setelah batch selesai, jalankan sel **hentikan tcpdump**; (4) unduh `/content/capture_batch.pcap`. Panduan penuh dan variasi lingkungan: **`METODOLOGI_FORENSIK_JARINGAN.md`** (unggah ke Colab atau baca di laptop). Jika `tcpdump` gagal (izin antarmuka), rekam PCAP di mesin lokal dengan perintah yang sama terhadap port Flask.


In [ ]:
# Opsi — mulai rekaman PCAP ke /content/capture_batch.pcap (loopback, port 5001).
# Lewati sel ini jika tidak butuh bukti paket; atau jika error, ikuti METODOLOGI_FORENSIK_JARINGAN.md di mesin lokal.
import os
import shutil
import subprocess

CAP = "/content/capture_batch.pcap"
PIDFILE = "/content/tcpdump_forensik.pid"
if os.path.isfile(PIDFILE):
    raise RuntimeError("tcpdump mungkin sudah jalan. Jalankan dulu sel 'hentikan tcpdump' atau hapus PID.")

if not shutil.which("tcpdump"):
    raise FileNotFoundError("tcpdump tidak terpasang. Jalankan ulang sel instalasi.")
with open(os.devnull, "wb") as devnull:
    proc = subprocess.Popen(
        ["tcpdump", "-i", "lo", "-w", CAP, "tcp", "port", "5001"],
        stdout=devnull,
        stderr=devnull,
    )
with open(PIDFILE, "w", encoding="utf-8") as f:
    f.write(str(proc.pid))
print("tcpdump PID", proc.pid, "→", CAP)
print("Lanjut: jalankan sel orkestrasi (ngrok+batch). Setelah itu: sel hentikan tcpdump, lalu unduh PCAP.")


In [ ]:
# Orkestrasi: tunnel ngrok + batch 28 sampel + ekspor dataset
# Jika memakai PCAP: pastikan sel tcpdump “mulai” sudah dijalankan sebelum sel ini.
import importlib.util
import os
import sys
from getpass import getpass
from IPython.display import HTML, display

# Colab menyimpan unggahan di /content; pastikan modul dataset bisa di-resolve
if "/content" not in sys.path:
    sys.path.insert(0, "/content")

path_py = "/content/source_code_forensik.py"
path_dataset = "/content/dataset_eksperimen.py"
for p in (path_py, path_dataset):
    if not os.path.isfile(p):
        raise FileNotFoundError(
            f"File tidak ada: {p}\nJalankan urutan: sel unggah source_code_forensik.py, lalu sel unggah dataset_eksperimen.py (masing-masing terpisah)."
        )

spec = importlib.util.spec_from_file_location("forensik", path_py)
forensik = importlib.util.module_from_spec(spec)
spec.loader.exec_module(forensik)

token = getpass("NGROK_AUTHTOKEN (dashboard ngrok):")
if not token.strip():
    raise ValueError("Token kosong.")

public_url = forensik.jalankan_colab_dengan_ngrok(token.strip(), port=5001, jalankan_simulasi=True)

chat_url = public_url + "/"
dash_url = public_url + "/dataset"
board_url = public_url + "/dashboard"
display(HTML(f"""
<section style="font-family:system-ui;padding:18px;background:#12121c;color:#eaeaf0;border-radius:14px;border:1px solid #2a2a3a;max-width:640px">
  <h3 style="margin:0 0 14px;color:#ff6b4a">Sesi publik (ngrok)</h3>
  <p><strong>Chat</strong> — <a href="{chat_url}" target="_blank" style="color:#7ec8e3">{chat_url}</a></p>
  <p><strong>Dashboard forensik</strong> — <a href="{board_url}" target="_blank" style="color:#7ec8e3">{board_url}</a></p>
  <p><strong>Dataset &amp; unduhan JSONL/CSV</strong> — <a href="{dash_url}" target="_blank" style="color:#7ec8e3">{dash_url}</a></p>
  <p style="opacity:.75;font-size:.88rem;margin-top:12px">Batch menjalankan 28 skenario; artefak lokal: <code>hasil_serangan.json</code>, <code>laporan_forensik.json</code>, folder <code>dataset/</code>.</p>
</section>
"""))

In [ ]:
# Hentikan tcpdump (opsional) — jalankan SETELAH batch/ngrok selesai, lalu unduh capture_batch.pcap dari Files.
import os
import signal

PIDFILE = "/content/tcpdump_forensik.pid"
if not os.path.isfile(PIDFILE):
    print("Tidak ada file PID — tcpdump tidak dimulai dari sel ini, atau sudah dihentikan.")
else:
    with open(PIDFILE, encoding="utf-8") as f:
        pid = int(f.read().strip())
    try:
        os.kill(pid, signal.SIGTERM)
    except ProcessLookupError:
        pass
    os.remove(PIDFILE)
    print("tcpdump dihentikan. Unduh berkas: /content/capture_batch.pcap (jika dibuat).")
